In [1]:
import os
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import pybullet as p
import pybullet_data

import ressources.affiche as affiche
import ressources.robotcontrol as robotcontrol
import modelisation.mgd as mgd
import modelisation.jacobienne as jacobienne

In [2]:
dt = 0.001

try:
    p.disconnect()
except Exception:
    pass

p.connect(p.GUI)
p.setGravity(0, 0, -9.81)
p.setPhysicsEngineParameter(fixedTimeStep=dt, numSubSteps=1)
p.resetDebugVisualizerCamera(2.0, 50, -35, (0.0, 0.0, 0.0))
p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)
p.setRealTimeSimulation(1)

p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.loadURDF("plane.urdf")

urdf_path = "ressources/staubli/staubli_tx60_support/urdf/tx60l.urdf"
robot = p.loadURDF(urdf_path, [0, 0, 0], [0, 0, 0, 1], flags=p.URDF_USE_INERTIA_FROM_FILE, useFixedBase=True)

nj = p.getNumJoints(robot)
nja = nj - 3
control_joints = list(range(0, nja))
end_eff_index = control_joints[-1]


In [3]:
frame_base = affiche.draw_coordinate_frame([0, 0, 0], p.getQuaternionFromEuler([0, 0, 0]), length=1.0)
frame_ee = []

q_init = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
robotcontrol.setJointPosition(robot, control_joints, q_init, kp=1.0, kv=1.0)
frame_ee = affiche.update_joint_frame(robot, end_eff_index, frame_ee, 0.1)

Joint position controller


In [4]:
link_state = p.getLinkState(robot, end_eff_index)
pos_sim = np.array(link_state[0])
rot_sim = np.array(p.getMatrixFromQuaternion(link_state[1])).reshape(3, 3)

T_mgd = mgd.mgd(q_init)
pos_mgd = mgd.mgd_Xp(T_mgd)
rot_mgd = mgd.mgd_Xr(T_mgd)

print("Position PyBullet :", np.round(pos_sim, 5))
print("Position MGD      :", np.round(pos_mgd, 5))
print("\nRotation PyBullet :\n", np.round(rot_sim, 5))
print("\nRotation MGD :\n", np.round(rot_mgd, 5))


Position PyBullet : [-2.30000e-04  2.00000e-02  1.28902e+00]
Position MGD      : [0.    0.02  1.295]

Rotation PyBullet :
 [[ 1. -0.  0.]
 [ 0.  1. -0.]
 [-0.  0.  1.]]

Rotation MGD :
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [5]:
T_total = 5.0

depart = [
    [0.3, 0.0, 0.0],
    [0.02, 0.0, 0.0],
    [0.8, 0.0, 0.0]
]

arrivee = [
    [0.9, 0.25, 0.0],
    [0.02, 0.0, 0.0],
    [0.1, 0.0, 0.0]
]

xbut = [arrivee[0][0], arrivee[1][0], arrivee[2][0]]